In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/yeonjiji/separation-data/test.jsonl
/kaggle/input/datasets/yeonjiji/separation-data/validate.jsonl
/kaggle/input/datasets/yeonjiji/separation-data/train.jsonl


In [2]:
%env CUDA_VISIBLE_DEVICES=0

%pip uninstall -y torchao

%pip install -q \
    transformers==5.16.1 \
    trl==1.12.0 \
    peft==0.20.0 \
    datasets \
    accelerate

env: CUDA_VISIBLE_DEVICES=0
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 102.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 91.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
# import importlib.util

# print("torchao installed:", importlib.util.find_spec("torchao") is not None)

In [4]:
import torch
import transformers
import trl
import peft

print("GPU:", torch.cuda.get_device_name(0))
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)

GPU: Tesla T4
Transformers: 5.16.1
TRL: 1.12.0
PEFT: 0.20.0


In [5]:
from datasets import load_dataset

DATA_DIR = "/kaggle/input/datasets/yeonjiji/separation-data"

data_files = {
    "train": f"{DATA_DIR}/train.jsonl",
    "validation": f"{DATA_DIR}/validate.jsonl",
    "test": f"{DATA_DIR}/test.jsonl",
}

raw_dataset = load_dataset("json", data_files=data_files)

raw_dataset

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 1602
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 400
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 501
    })
})

In [6]:
raw_dataset["train"][0]

{'messages': [{'role': 'system',
   'content': 'You are an expert liquid-liquid extraction chemist.\nUse the provided experiment description to predict the extraction selectivity.\nReturn only VL, L, U, M, H.'},
  {'role': 'user',
   'content': '0.5 M of CCCCCCN(CCCCCC)C(=O)CCCCC in toluene is used to extract 50 mM Th(IV) from an aqueous solution of 4 M HNO3 and 4 M nitrate at 20 C. The contact time is 10 min and the radiolytic dosage is 0 kGy.'},
  {'role': 'assistant', 'content': 'L'}]}

In [7]:
from collections import Counter

for split in ["train", "validation", "test"]:
    labels = [
        x["messages"][-1]["content"].strip()
        for x in raw_dataset[split]
    ]

    print(split, Counter(labels))

train Counter({'U': 525, 'L': 431, 'M': 309, 'VL': 180, 'H': 157})
validation Counter({'U': 139, 'L': 83, 'M': 81, 'VL': 54, 'H': 43})
test Counter({'U': 173, 'L': 125, 'M': 109, 'VL': 52, 'H': 42})


In [8]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [9]:
LABELS = ["VL", "L", "U", "M", "H"]

for label in LABELS:
    ids = tokenizer.encode(label, add_special_tokens=False)
    print(f"{label:2s} -> {ids} ({len(ids)} token)")

VL -> [30698] (1 token)
L  -> [43] (1 token)
U  -> [52] (1 token)
M  -> [44] (1 token)
H  -> [39] (1 token)


In [10]:
# turn off the thinking mode
def format_example(example):
    messages = example["messages"]

    # system + user
    prompt_messages = messages[:-1]

    # VL/L/U/M/H -> answer
    label = messages[-1]["content"].strip()

    prompt = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    completion = label + tokenizer.eos_token

    return {
        "prompt": prompt,
        "completion": completion,
    }

dataset = raw_dataset.map(
    format_example,
    remove_columns=["messages"],
)

dataset["train"][0]

Map:   0%|          | 0/1602 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/501 [00:00<?, ? examples/s]

{'prompt': '<|im_start|>system\nYou are an expert liquid-liquid extraction chemist.\nUse the provided experiment description to predict the extraction selectivity.\nReturn only VL, L, U, M, H.<|im_end|>\n<|im_start|>user\n0.5 M of CCCCCCN(CCCCCC)C(=O)CCCCC in toluene is used to extract 50 mM Th(IV) from an aqueous solution of 4 M HNO3 and 4 M nitrate at 20 C. The contact time is 10 min and the radiolytic dosage is 0 kGy.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n',
 'completion': 'L<|im_end|>'}

In [11]:
import numpy as np

lengths = []

for x in dataset["test"]:
    text = x["prompt"] + x["completion"]

    length = len(
        tokenizer(
            text,
            add_special_tokens=False
        )["input_ids"]
    )

    lengths.append(length)

print("Max:", max(lengths))
print("95 percentile:", np.percentile(lengths, 95))
print("Mean:", np.mean(lengths))

Max: 151
95 percentile: 144.0
Mean: 136.34930139720558


In [12]:
MAX_LENGTH = 256

In [13]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],

    bias="none",
    task_type="CAUSAL_LM",
)

In [14]:
from trl import SFTConfig

OUTPUT_DIR = "/kaggle/working/separationllm-qwen3-1.7b-lora"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    # training
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,

    learning_rate=1e-4,
    warmup_steps=30,

    # T4
    fp16=True,
    bf16=False,

    gradient_checkpointing=True,
    use_cache=False,

    # dataset
    max_length=MAX_LENGTH,
    packing=False,
    completion_only_loss=True,

    # validation / saving
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_steps=20,
    report_to="none",

    seed=2024,

    model_init_kwargs={
        "dtype": torch.float16,
    },
)

In [15]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=MODEL_NAME,

    args=training_args,

    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],

    processing_class=tokenizer,

    peft_config=lora_config,
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Adding EOS to train dataset:   0%|          | 0/1602 [00:00<?, ? examples/s]

...

Adding EOS to eval dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

In [16]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.658562,0.764645,0.551804,219050.000000,0.685000
2,0.578961,0.595211,0.519899,438100.000000,0.747500
3,0.448842,0.537695,0.425345,657150.000000,0.765000


TrainOutput(global_step=603, training_loss=0.6324602841935545, metrics={'train_runtime': 962.9514, 'train_samples_per_second': 4.991, 'train_steps_per_second': 0.626, 'total_flos': 5703081219403776.0, 'train_loss': 0.6324602841935545, 'epoch': 3.0})

In [17]:
FINAL_DIR = "/kaggle/working/separationllm-qwen3-1.7b-lora-final"

trainer.model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print("Saved to:", FINAL_DIR)

Saved to: /kaggle/working/separationllm-qwen3-1.7b-lora-final
